# Student Performance Classification with Logistic Regression

In [1]:
import json
import joblib
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv('student_performance_dataset.csv')
df.head()

,student_id,age,gender,city_type,study_hours_per_day,deep_work_sessions,assignment_completion_rate,attendance_percentage,social_media_hours,doomscrolling_before_sleep,...,family_support,financial_stress,learning_style,career_goal,productivity_after_midnight,revision_efficiency,burnout_risk,consistency_score,final_exam_score,performance_category
0,0,21,Female,Semi-Urban,3.2,7,100,70,3.8,0,...,10,6,Audio,Engineering,9,1,10,3,32,Low
1,1,19,Female,Semi-Urban,3.9,2,46,70,2.6,1,...,2,6,Practical,Business,6,10,4,6,59,Medium
2,2,16,Female,Urban,4.3,7,54,57,4.3,1,...,1,9,Visual,Engineering,7,9,8,1,34,Low
3,3,19,Male,Semi-Urban,5.3,1,78,90,1.7,0,...,5,10,Reading,Medical,7,7,9,10,60,Medium
4,4,17,Female,Urban,4.1,3,100,81,2.6,0,...,5,6,Visual,Medical,4,10,7,9,77,High


In [3]:
features = [
    'age', 'gender', 'city_type', 'study_hours_per_day', 'sleep_hours',
    'stress_level', 'motivation_level', 'focus_score',
    'attendance_percentage', 'assignment_completion_rate'
]
target = 'performance_category'

X = df[features].copy()
y = df[target].copy()

num_cols = X.select_dtypes(include=['number']).columns
cat_cols = X.select_dtypes(exclude=['number']).columns
X[num_cols] = X[num_cols].fillna(X[num_cols].median())
for c in cat_cols:
    if X[c].isna().any():
        X[c] = X[c].fillna(X[c].mode().iloc[0])

X = pd.get_dummies(X, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [4]:
model = LogisticRegression(max_iter=2000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='macro')

print(f'Accuracy: {acc:.4f}')
print(f'Macro F1: {f1:.4f}')
print(classification_report(y_test, y_pred))

Accuracy: 0.5433
Macro F1: 0.4768
              precision    recall  f1-score   support

        High       0.55      0.19      0.28        89
         Low       0.58      0.59      0.59       239
      Medium       0.51      0.62      0.56       272

    accuracy                           0.54       600
   macro avg       0.55      0.47      0.48       600
weighted avg       0.55      0.54      0.53       600



c:\Users\abhin\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [5]:
joblib.dump(model, 'logistic_regression_model.pkl')

metadata = {
    'feature_columns': list(X.columns),
    'class_names': list(model.classes_),
    'accuracy': round(float(acc), 4),
    'f1_macro': round(float(f1), 4)
}

with open('model_metadata.json', 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2)

print('Saved logistic_regression_model.pkl and model_metadata.json')

Saved logistic_regression_model.pkl and model_metadata.json
